# Data Preprocessing and Feature Engineering
## 10-Year Coronary Heart Disease Risk Prediction

This notebook prepares the cardiovascular dataset for machine-learning model development.

The preprocessing decisions are based on the findings from the EDA stage.

The main steps include:

- handling missing target values
- separating input features and target
- splitting the data into training and testing sets
- handling missing feature values
- encoding categorical variables
- investigating and treating outliers where necessary
- feature engineering and feature selection
- scaling numerical variables where required
- handling class imbalance

Care will be taken to avoid data leakage during preprocessing.

In [0]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import RobustScaler




In [0]:
df = pd.read_csv(
    "/Volumes/workspace/default/fdm_project/heart.csv"
)

df.head()

In [0]:
print("Original dataset shape:", df.shape)

In [0]:
print("Missing target values before removal:")
print(df["TenYearCHD"].isnull().sum())

In [0]:
df = df.dropna(subset=["TenYearCHD"]).copy()

In [0]:
print("Missing target values after removal:")
print(df["TenYearCHD"].isnull().sum())

print("Dataset shape after removing missing targets:")
print(df.shape)

### Observation

The target variable `TenYearCHD` contained 361 missing values.

Since `TenYearCHD` is the class label, these missing values were not imputed because this would require guessing the correct outcome.

Therefore, records with missing target values were removed.

After removing these records, 10,639 labelled records remain for model development.

In [0]:
df["TenYearCHD"] = df["TenYearCHD"].astype(int)

In [0]:
df["TenYearCHD"].value_counts()

we converted the target variable from floating-point values (`0.0` and `1.0`) to integer class labels (`0` and `1`) for easier classification and interpretation.

In [0]:
X = df.drop(columns=["TenYearCHD"])
y = df["TenYearCHD"]

print("X shape:", X.shape)
print("y shape:", y.shape)

%md
## 2. Train-Test Split and Leakage Prevention

The dataset was divided into training and testing sets using an 80:20 split.


The split was performed before fitting preprocessing methods such as imputation, scaling, outlier treatment, or resampling.

This helps prevent data leakage because information from the test set should not influence the training process.

In [0]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [0]:
print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)

In [0]:
print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True) * 100)

print("\nTesting target distribution:")
print(y_test.value_counts(normalize=True) * 100)

In [0]:
df.shape
X_train.shape
X_test.shape
y_train.value_counts(normalize=True) * 100
y_test.value_counts(normalize=True) * 100

### Observation

After removing the records with missing target values, 10,639 labelled records remained.

The data was divided into 80% training data and 20% testing data.

- Training set: 8,511 records
- Testing set: 2,128 records

A stratified split was used because the target variable is imbalanced. The class distribution is almost the same in both the training and testing sets.

This helps ensure that both sets represent the original target distribution fairly.

%md
## 3. Handling Missing Feature Values

The EDA showed that several input features contain missing values.

For numerical features, median imputation will be used. Several numerical features contain skewness and extreme values, so the median is less affected by these values than the mean.

For categorical features, the most frequent category will be used to replace missing values.

The imputation values will be learned only from the training data and then applied to the testing data to prevent data leakage.

In [0]:
# Numerical features
numerical_features = [
    "age",
    "cigsPerDay",
    "totChol",
    "sysBP",
    "diaBP",
    "BMI",
    "heartRate",
    "glucose"
]

# Categorical features
categorical_features = [
    "gender",
    "currentSmoker",
    "BPMeds",
    "prevalentStroke",
    "prevalentHyp",
    "diabetes"
]

# Ordinal feature
ordinal_features = ["education"]

In [0]:
# Missing values in the training set

X_train.isnull().sum()

In [0]:
from sklearn.impute import SimpleImputer

In [0]:
# Median imputer for numerical features
numeric_imputer = SimpleImputer(strategy="median")

# Most frequent value imputer for categorical features
categorical_imputer = SimpleImputer(strategy="most_frequent")

In [0]:
X_train_clean = X_train.copy()
X_test_clean = X_test.copy()

In [0]:
# Learn median values from training data
# and apply them to training data

X_train_clean[numerical_features] = numeric_imputer.fit_transform(
    X_train[numerical_features]
)

# Apply the SAME learned median values to testing data

X_test_clean[numerical_features] = numeric_imputer.transform(
    X_test[numerical_features]
)

In [0]:
# Learn the most frequent categories from training data

X_train_clean[categorical_features] = categorical_imputer.fit_transform(
    X_train[categorical_features]
)

# Apply the same values to testing data

X_test_clean[categorical_features] = categorical_imputer.transform(
    X_test[categorical_features]
)

In [0]:
print("Missing values in training set:")
print(X_train_clean.isnull().sum())

print("\nTotal missing values in training set:")
print(X_train_clean.isnull().sum().sum())

print("\nTotal missing values in testing set:")
print(X_test_clean.isnull().sum().sum())

%md
### Observation

The missing values in the input features were handled using imputation.

- Numerical missing values were replaced using the median value calculated from the training data.
- Categorical missing values were replaced using the most frequent category from the training data.
- The same learned values were then applied to the testing data.

After imputation, no missing values remained in the training or testing feature sets.

Using only the training data to learn the imputation values helps prevent data leakage.

%md
## 4. Encoding Categorical Variables

Some features in the dataset contain categorical text values such as `Male`, `Female`, `Yes`, and `No`.

These values need to be converted into numerical form before they can be used by machine-learning algorithms.

Since these variables contain only two categories, we used binary encoding.

In [0]:
for column in categorical_features:
    print(column, X_train_clean[column].unique())

In [0]:
yes_no_columns = [
    "currentSmoker",
    "BPMeds",
    "prevalentStroke",
    "prevalentHyp",
    "diabetes"
]

for column in yes_no_columns:
    X_train_clean[column] = X_train_clean[column].map({
        "No": 0,
        "Yes": 1
    })

    X_test_clean[column] = X_test_clean[column].map({
        "No": 0,
        "Yes": 1
    })

In [0]:
X_train_clean["gender"] = X_train_clean["gender"].map({
    "Female": 0,
    "Male": 1
})

X_test_clean["gender"] = X_test_clean["gender"].map({
    "Female": 0,
    "Male": 1
})

In [0]:
X_train_clean.head()

In [0]:
X_train_clean.dtypes

In [0]:
print("Missing values after encoding:")
print(X_train_clean.isnull().sum().sum())
print(X_test_clean.isnull().sum().sum())

%md
### Observation

The categorical variables were successfully converted into numerical form.

- `Male` was encoded as `1` and `Female` as `0`.
- For the Yes/No variables, `Yes` was encoded as `1` and `No` as `0`.
- The `education` variable was kept as an ordinal feature because its values already represent ordered levels.

After encoding, all input features are numerical and there are no missing values in either the training or testing datasets.

In [0]:
X_train_clean["education"] = X_train_clean["education"].astype(int)
X_test_clean["education"] = X_test_clean["education"].astype(int)

## 5. Outlier Treatment

During EDA, we identified several possible outliers in numerical features such as `cigsPerDay`, `totChol`, `sysBP`, `diaBP`, `BMI`, `heartRate`, and `glucose`.

Since these are medical measurements, we did not remove the outliers automatically. Some extreme values may represent genuine patient conditions rather than data-entry errors.

We decided to keep these values unless there is clear evidence that they are invalid. This also helps us avoid removing important high-risk patient records.

For models that require scaling, we will later select a suitable scaling method while considering the effect of these extreme values.

In [0]:
outlier_check = X_train_clean[
    [
        "cigsPerDay",
        "totChol",
        "sysBP",
        "diaBP",
        "BMI",
        "heartRate",
        "glucose"
    ]
].agg(["min", "max"])

display(outlier_check)

### Observation

We checked the minimum and maximum values of the numerical features to further examine the possible outliers identified during EDA.

We found several high values, such as:

- `cigsPerDay` up to 70
- `totChol` up to 696
- `sysBP` up to 295
- `diaBP` up to 142.5
- `BMI` up to 51.37
- `heartRate` up to 143
- `glucose` up to 394

Although these values are far from the common range of the dataset, we did not remove them automatically because they may represent genuine patient measurements.

Therefore, we decided to keep the outliers unless there is clear evidence that a value is invalid.

%md
## 6. Feature Engineering

During EDA, we found that `sysBP` and `diaBP` have a strong positive correlation.

Therefore, we created a new feature called `pulsePressure`, which represents the difference between systolic and diastolic blood pressure.

Pulse pressure was calculated as:

`pulsePressure = sysBP - diaBP`

We created this feature because it combines information from both blood-pressure measurements into one additional feature that may be useful for predicting CHD.

We will later compare model performance to determine whether this new feature improves the prediction results.

In [0]:
# Create pulse pressure feature

X_train_clean["pulsePressure"] = (
    X_train_clean["sysBP"] - X_train_clean["diaBP"]
)

X_test_clean["pulsePressure"] = (
    X_test_clean["sysBP"] - X_test_clean["diaBP"]
)

In [0]:
X_train_clean[
    ["sysBP", "diaBP", "pulsePressure"]
].head()

In [0]:
print("Training shape:", X_train_clean.shape)
print("Testing shape:", X_test_clean.shape)

In [0]:
X_train_clean["pulsePressure"].describe()

### Observation

We successfully created the new `pulsePressure` feature using `sysBP` and `diaBP`.

The number of input features increased from 15 to 16.

We kept the original `sysBP` and `diaBP` features at this stage. During feature selection and model evaluation, we will check whether keeping all three blood-pressure-related features improves the model or creates unnecessary redundancy.

## 7. Feature Selection

We examined the input features to identify whether any features contain very similar information or provide very little variation.

Feature selection was performed using only the training data to avoid using information from the testing set.

We did not remove features only because they had a low correlation with the target, because some features may still be useful when combined with other variables or when using non-linear machine-learning models.

In [0]:
# Correlation between input features in the training data

feature_correlation = X_train_clean.corr()

plt.figure(figsize=(12, 8))

sns.heatmap(
    feature_correlation,
    cmap="coolwarm",
    center=0
)

plt.title("Correlation Between Input Features")
plt.show()

In [0]:
# Identify highly correlated feature pairs

corr_matrix = X_train_clean.corr().abs()

upper_triangle = corr_matrix.where(
    np.triu(
        np.ones(corr_matrix.shape),
        k=1
    ).astype(bool)
)

high_correlation_pairs = []

for column in upper_triangle.columns:
    for row in upper_triangle.index:
        value = upper_triangle.loc[row, column]

        if pd.notna(value) and value > 0.70:
            high_correlation_pairs.append(
                (row, column, round(value, 2))
            )

high_correlation_pairs

### Observation

We checked the training features for high correlations and identified three feature pairs with a correlation above 0.70.

- `currentSmoker` and `cigsPerDay` have a correlation of approximately 0.72.
- `sysBP` and `diaBP` have a correlation of approximately 0.74.
- `sysBP` and the newly created `pulsePressure` feature have a strong correlation of approximately 0.86.

We did not remove these features at this stage because they may still provide useful information to the machine-learning models.

We decided to keep them for the initial model development and later compare the model results to determine whether removing any highly correlated features improves performance.

## 8. Feature Scaling

We identified that the numerical features have different value ranges and some of them contain extreme values.

Therefore, we created a scaled version of the numerical features for machine-learning algorithms that are sensitive to feature scale.

We used `RobustScaler` because it is less affected by extreme values compared with standard scaling.

The scaler was fitted only on the training data and the same transformation was applied to the testing data to prevent data leakage.

We also kept the original unscaled data because tree-based algorithms such as Decision Tree and Random Forest do not require feature scaling.

In [0]:
features_to_scale = [
    "age",
    "cigsPerDay",
    "totChol",
    "sysBP",
    "diaBP",
    "BMI",
    "heartRate",
    "glucose",
    "pulsePressure"
]

In [0]:
X_train_scaled = X_train_clean.copy()
X_test_scaled = X_test_clean.copy()

In [0]:
scaler = RobustScaler()

X_train_scaled[features_to_scale] = scaler.fit_transform(
    X_train_clean[features_to_scale]
)

X_test_scaled[features_to_scale] = scaler.transform(
    X_test_clean[features_to_scale]
)

In [0]:
X_train_scaled[features_to_scale].head()

In [0]:
print("Scaled training shape:", X_train_scaled.shape)
print("Scaled testing shape:", X_test_scaled.shape)

### Observation

We successfully scaled the numerical features using `RobustScaler`.

The number of records and features remained unchanged after scaling.

We kept both scaled and unscaled versions of the dataset so that we can use the appropriate version depending on the machine-learning algorithm.

The scaler was fitted only using the training data, which prevents information from the testing data from affecting the preprocessing process.

%md
## 9. Handling Class Imbalance

During EDA, we identified that the target variable `TenYearCHD` is imbalanced.

Around 85% of the labelled records belong to class `0`, while only around 15% belong to class `1`.

We did not modify the testing data because it should represent the original unseen data distribution.

At this stage, we also decided not to apply oversampling directly to the full training dataset. During model development, we will compare suitable methods such as class weighting and SMOTE and evaluate whether they improve the detection of the minority CHD-positive class.

If SMOTE is used, it will only be applied to the training data and within the model-training process to avoid data leakage.

In [0]:
print("Training class distribution:")

print(y_train.value_counts())
print()

print(y_train.value_counts(normalize=True) * 100)

In [0]:
import matplotlib.pyplot as plt

target_counts = y_train.value_counts().sort_index()

plt.figure(figsize=(6,4))
plt.bar(target_counts.index.astype(str), target_counts.values)
plt.title("Target Distribution (TenYearCHD)")
plt.xlabel("TenYearCHD")
plt.ylabel("count")
plt.show()

## 10 Target Class Distribution After Preprocessing

We examined the class distribution of the target variable after removing missing target values.

This helps us confirm whether class imbalance is still present before model development.

## 11. Final Preprocessing Summary

We completed the main preprocessing and feature-engineering steps required before model development.

The main steps we completed were:

-  removed 361 records with missing `TenYearCHD` values because the target class should not be guessed or imputed.
-  converted the target variable from floating-point values to integer class labels.
-  separated the input features and target variable.
-  divided the data into 80% training data and 20% testing data using a stratified split.
-  handled missing numerical values using median imputation.
-  handled missing categorical values using the most frequent category.
-  learned the imputation values only from the training data and applied the same values to the testing data to avoid data leakage.
-  converted categorical features into numerical form using binary encoding.
-  investigated possible outliers and decided not to remove them automatically because some extreme values may represent genuine patient measurements.
-  created a new feature called `pulsePressure` using systolic and diastolic blood pressure.
-  investigated highly correlated and low-variation features but decided to keep all features for the initial model development.
-  created both scaled and unscaled versions of the data. `RobustScaler` was used for numerical features because the dataset contains several extreme values.
-  identified class imbalance in the target variable and decided to compare suitable methods such as class weighting and SMOTE during model development.
-  kept the testing data unchanged so that it can be used for fair evaluation on unseen data.

The preprocessed data is now ready for machine-learning model development.